# Cylinder 2D displays — emission-process comparison

Prediction-only renderings of a single muon track propagated through the SK
geometry, side-by-side for three emission configurations:

1. **Cherenkov only**
2. **Scintillation only**
3. **Both**

Same geometry, same `ParticleParams`, same `n_photons` budget. We start from
the standard SK water configs and use `medium_override` on
`setup_event_simulator` to swap in WbLS chemistry and flip the
`emission_processes` tuple per run — no new geom/physics/material files
needed.

In [ ]:
import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

from lucid.simulation import setup_event_simulator
from lucid.detector_params import ParticleParams
from lucid.wavelength.medium import make_medium
from lucid.geometry import generate_detector

In [ ]:
# Three simulators — same SK geometry + physics config in all three. The
# `medium_override` kwarg swaps WbLS chemistry in (so we get the WbLS
# scintillation values) and flips `emission_processes` per run. Setup-time
# dispatch then builds only the surrogates each mode needs.

GEOM_SK   = '../config/SK_geom_config.json'
PHYS_SK   = '../config/SK_physics_config.json'
N_PHOTONS = 1_000_000
K         = 7

m_wbls      = make_medium('wbls')
m_wbls_cher = m_wbls._replace(emission_processes=('cherenkov',))
m_wbls_sc   = m_wbls._replace(emission_processes=('scintillation',))
m_wbls_both = m_wbls   # default — ('cherenkov', 'scintillation')

common = dict(json_filename=GEOM_SK, physics_config=PHYS_SK,
              n_photons=N_PHOTONS, K=K, particle='muon',
              is_data=False, hit_mode='aggregated',
              default_detector_params=True)

sim_cher  = setup_event_simulator(**common, medium_override=m_wbls_cher)
sim_scint = setup_event_simulator(**common, medium_override=m_wbls_sc)
sim_both  = setup_event_simulator(**common, medium_override=m_wbls_both)

for name, sim in [('cherenkov', sim_cher),
                   ('scintillation', sim_scint),
                   ('both', sim_both)]:
    dp = sim.default_detector_params
    s_repr = 'nan' if jnp.isnan(dp.S) else f'{float(dp.S):.1f}'
    print(f'{name:14s}  qe={float(dp.qe):.3f}  S={s_repr}  '
          f'tau_2={float(dp.tau_2):.2f}  moyal_loc={float(dp.moyal_loc):.2f}')

detector = generate_detector(GEOM_SK)
NUM_DETECTORS = len(detector.all_points)
print(f'NUM_DETECTORS = {NUM_DETECTORS}')

In [ ]:
# Single track, evaluated through each simulator.
key = jax.random.PRNGKey(719007)

true_position  = jnp.array([-10., 0., 0.])
true_direction = jnp.array([1., 0., 0.])
true_energy    = jnp.array(1000.0)   # MeV — generic 1 GeV muon

true_track = ParticleParams.from_cartesian(
    energy=true_energy, position=true_position,
    direction=true_direction, t0=0.0,
)

pred_cher  = jax.lax.stop_gradient(sim_cher (true_track, key))
pred_scint = jax.lax.stop_gradient(sim_scint(true_track, key))
pred_both  = jax.lax.stop_gradient(sim_both (true_track, key))

for name, (charges, _) in [('cherenkov',     pred_cher),
                            ('scintillation', pred_scint),
                            ('both',          pred_both)]:
    pos = jnp.asarray(charges) > 0
    print(f'{name:14s}  total Q = {float(jnp.sum(charges)):.3e}  '
          f'hit channels = {int(jnp.sum(pos))} / {NUM_DETECTORS}')

In [ ]:
# 2D cylinder-unwrap detector display. Lifted from cylinder_2D_displays.ipynb
# — same renderer, operates on DENSE (charges, times) arrays.

from matplotlib.collections import EllipseCollection
from scipy.spatial.distance import pdist
from mpl_toolkits.axes_grid1 import make_axes_locatable


def _calculate_min_distance(positions):
    distances = pdist(positions)
    return np.min(distances) if len(distances) > 0 else 1.0


def create_detector_display(json_filename):
    detector = generate_detector(json_filename)
    radius = detector.r
    height = detector.H

    sensor_positions = np.array(detector.all_points)
    sensor_cases = np.array([detector.ID_to_case[i] for i in range(len(detector.all_points))])
    n_sensors = len(sensor_positions)

    def display(all_charges, all_times, *, file_name=None, plot_time=False,
                log_scale=False, vmin=None, vmax=None, perc_min=1, perc_max=99,
                title=None):
        all_charges = np.asarray(all_charges)
        all_times = np.asarray(all_times)
        all_values = all_times if plot_time else all_charges

        positive_values = all_values[all_values > 0]
        if len(positive_values) > 0:
            if vmin is None: vmin = np.percentile(positive_values, perc_min)
            if vmax is None: vmax = np.percentile(positive_values, perc_max)
        else:
            if vmin is None: vmin = 0.1 if log_scale else 0
            if vmax is None: vmax = 1
        if perc_min == 0 and not plot_time and not log_scale:
            vmin = 0.001
        elif perc_min == 0 and not plot_time and log_scale:
            vmin = 0.1

        if log_scale:
            plot_values = np.copy(all_values)
            plot_values[plot_values <= 0] = vmin
            plot_values = np.clip(plot_values, vmin, vmax)
            cmap = plt.get_cmap('viridis_r') if plot_time else plt.get_cmap('plasma')
            norm = plt.matplotlib.colors.LogNorm(vmin=vmin, vmax=vmax)
        else:
            plot_values = np.clip(np.copy(all_values), vmin, vmax)
            cmap = plt.get_cmap('viridis_r') if plot_time else plt.get_cmap('viridis')
            norm = plt.Normalize(vmin=vmin, vmax=vmax)
        color_gradient = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

        caps_offset = 1.05 * height / 2 + radius
        x = np.zeros(n_sensors); y = np.zeros(n_sensors)
        barrel = sensor_cases == 0
        theta = np.arctan2(sensor_positions[barrel, 1], sensor_positions[barrel, 0])
        theta = (theta + np.pi * 3 / 2) % (2 * np.pi) / 2
        x[barrel] = theta * radius * 2
        y[barrel] = sensor_positions[barrel, 2]
        top = sensor_cases == 1
        x[top] = sensor_positions[top, 0] + np.pi * radius
        y[top] = caps_offset + sensor_positions[top, 1]
        bot = sensor_cases == 2
        x[bot] = sensor_positions[bot, 0] + np.pi * radius
        y[bot] = -caps_offset - sensor_positions[bot, 1]

        transformed = np.column_stack((x, y))
        circle_diameter = _calculate_min_distance(transformed)
        x_min, x_max = np.min(x) - circle_diameter, np.max(x) + circle_diameter
        y_min, y_max = np.min(y) - circle_diameter, np.max(y) + circle_diameter
        x_range = x_max - x_min; y_range = y_max - y_min
        fig_w = 8; fig_h = fig_w * (y_range / x_range)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor='white')

        colors = color_gradient.to_rgba(plot_values)
        zero_mask = all_values <= 0
        colors[zero_mask] = np.array([0.9, 0.9, 0.9, 1.0])

        ells = EllipseCollection(
            widths=circle_diameter, heights=circle_diameter, angles=0, units='x',
            facecolors=colors, offsets=transformed,
            transOffset=ax.transData, edgecolors='none',
        )
        ax.add_collection(ells)
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        ax.set_aspect('equal', adjustable='box')
        ax.axis('off')
        if title:
            ax.set_title(title, fontsize=14)

        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='5%', pad=0.1)
        cbar = plt.colorbar(color_gradient, cax=cax)
        label = 'Time (ns)' if plot_time else 'Photoelectron Count (a.u.)'
        cbar.set_label(label + (' (log scale)' if log_scale else ''), fontsize=14)
        plt.tight_layout()
        if file_name:
            plt.savefig(file_name, bbox_inches='tight', pad_inches=0.1,
                        facecolor='white', edgecolor='none')
        plt.show()

    return display


detector_display = create_detector_display(GEOM_SK)

In [ ]:
figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

predictions = [
    ('cherenkov',     pred_cher),
    ('scintillation', pred_scint),
    ('both',          pred_both),
]

for tag, (charges, times) in predictions:
    print(f'\n=== {tag} ===')
    detector_display(charges, times,
                     file_name=f'figures/pred_{tag}_charge.pdf',
                     plot_time=False, perc_min=0.0, log_scale=False,
                     title=f'{tag} — charge')
    detector_display(charges, times,
                     file_name=f'figures/pred_{tag}_time.pdf',
                     plot_time=True, perc_min=0.0, perc_max=100.0,
                     title=f'{tag} — time')